[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/InterstellarNerd8760/lunar-atlas/blob/proxima/terrain-notebooks/notebooks/prepare-south-pole-sites.ipynb)

# LOLA south pole sites: 5 m terrain for landing areas

NASA Goddard (PGDA) made 5 m/pixel LOLA elevation maps for candidate south-pole landing sites. Each file is a square in **south polar stereographic** coordinates: x/y in meters from the pole, like a flat map looking down on the pole.

For each site this notebook writes, in the same format as `public/data/global-height`:
- `<site>-height.f32` + `.json`: the site, averaged down to `MAX_PX` pixels wide.
- `<site>-context.f32` + `.json` (optional): a wider, coarser box from the 80 m polar LOLA map, so far-away ridges can block the Sun on the horizon.

Each JSON has the site center in lat/lon, the box in meters and degrees, and the resolution.

Runs on a free CPU runtime in Colab or Kaggle (on Kaggle, turn on **Internet**). **Runtime → Run all.**

In [ ]:
%pip -q install rasterio matplotlib requests

## 1. Settings
`SITE_IDS` starts with PGDA's nine numbered sites. The catalog in step 3 has 27 sites; add any ID from it. Several overlap NASA's 2024 Artemis III regions (Haworth, Malapert massif, Nobile rims 1 and 2 as `Site06` and `DM2`, and Leibnitz beta plateau, now called Mons Mouton).

In [ ]:
SITE_IDS = ["Site01", "Site04", "Site06", "Site07", "Site11", "Site20", "Site20v2", "Site23", "Site42"]
MAX_PX = 1024            # output width in pixels. A 16 km site at 1024 px is about 15.6 m/px
CROP_KM = None           # None = whole site. A number = square box this many km wide around the center
INCLUDE_CONTEXT = True   # also export the wider horizon box from the 80 m polar map
CONTEXT_KM = 150         # width of the horizon box in km
CONTEXT_PX = 512         # width of the horizon box in pixels (150 km / 512 px is about 290 m/px)
KEEP_RAW = True          # False deletes each 40-140 MB source file after use to save disk
ASSET_NAME = "lola-sites"
USE_DRIVE = False        # True: save exports to Google Drive (Colab only)
DOWNLOAD_ZIP = True      # Colab: download the zip at the end

## 2. Setup
Helper functions. Nothing to change here.

In [ ]:
import json
import sys
import time
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rasterio.warp
import requests
from rasterio.enums import Resampling
from rasterio.windows import from_bounds

IN_COLAB = "google.colab" in sys.modules

# Raw downloads always stay on the runtime disk. Only finished exports go to Drive.
RAW = Path("data/raw")
OUT_ROOT = Path("data/work")
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/lunar-atlas")
EXPORT = OUT_ROOT / "exports" / ASSET_NAME
RAW.mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)

MOON_RADIUS_M = 1737400
# Moon lat/lon (same sphere as the global-height export).
MOON_GEOGRAPHIC = rasterio.crs.CRS.from_proj4("+proj=longlat +R=1737400 +no_defs")


def remote_size(url):
    """Ask the server how big a file is (HTTP HEAD) without downloading it."""
    response = requests.head(url, allow_redirects=True, timeout=30)
    response.raise_for_status()
    return int(response.headers.get("Content-Length", 0))


def download(url, name=None):
    """Download once into data/raw. Re-running the cell skips finished files."""
    path = RAW / (name or Path(url).name)
    if path.exists():
        return path
    temporary_path = path.with_suffix(path.suffix + ".part")
    with requests.get(url, stream=True, timeout=(15, 120)) as response:
        response.raise_for_status()
        with temporary_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)
    temporary_path.replace(path)
    return path


def save_grid(elevation, metadata, folder, name):
    """Write <name>.f32 (float32 little-endian, row-major) and <name>.json."""
    elevation = np.asarray(elevation, dtype="<f4")
    if not np.isfinite(elevation).any():
        raise ValueError(f"{name}: no valid elevations found.")
    folder.mkdir(parents=True, exist_ok=True)
    metadata = dict(metadata)
    metadata["minElevationM"] = float(np.nanmin(elevation))
    metadata["maxElevationM"] = float(np.nanmax(elevation))
    elevation.tofile(folder / f"{name}.f32")
    (folder / f"{name}.json").write_text(json.dumps(metadata, indent=2) + "\n")
    return metadata


def zip_folder(folder, zip_path):
    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as bundle:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                bundle.write(path, path.relative_to(folder.parent))
    return zip_path


def folder_size_mb(folder):
    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) / 1e6

print("Colab" if IN_COLAB else "Not Colab (Kaggle or local)", "| exports go to", EXPORT.resolve())


def polar_metadata(transform, width, height, crs, extra):
    """Metadata for a south-polar-stereographic grid, plus lat/lon facts."""
    left, top = transform * (0, 0)
    right, bottom = transform * (width, height)
    center_x, center_y = transform * (width / 2, height / 2)
    lon, lat = rasterio.warp.transform(crs, MOON_GEOGRAPHIC, [center_x], [center_y])
    # Sample the outline to find the lat/lon box the grid covers.
    edge = np.linspace(0, 1, 50)
    xs = np.concatenate([left + (right - left) * edge, np.full(50, right),
                         left + (right - left) * edge, np.full(50, left)])
    ys = np.concatenate([np.full(50, top), top + (bottom - top) * edge,
                         np.full(50, bottom), top + (bottom - top) * edge])
    lons, lats = rasterio.warp.transform(crs, MOON_GEOGRAPHIC, xs.tolist(), ys.tolist())
    pole_inside = left <= 0 <= right and bottom <= 0 <= top
    metadata = {
        **extra,
        "width": width, "height": height, "encoding": "float32-le",
        "units": "m", "order": "row-major", "nodataEncoding": "NaN",
        "referenceRadiusM": MOON_RADIUS_M,
        "globalExtentDegrees": None,
        "projection": "south polar stereographic (true scale at the pole), meters",
        "crsWkt": crs.to_wkt(),
        "affine": list(transform)[:6],
        "extentM": [left, bottom, right, top],
        "resolutionM": abs(transform.a),
        "centerLatLon": [round(lat[0], 6), round(((lon[0] + 180) % 360) - 180, 6)],
        "latRange": [-90.0 if pole_inside else round(min(lats), 6), round(max(lats), 6)],
        "lonRange": [-180.0, 180.0] if pole_inside else [round(min(lons), 6), round(max(lons), 6)],
    }
    return metadata


def read_grid(path, bounds, out_px, scale=1.0, offset=0.0):
    """Read a box (left, bottom, right, top in meters), averaged down to out_px wide."""
    with rasterio.open(path) as dataset:
        left, bottom, right, top = bounds
        # Keep the box inside the file.
        b = dataset.bounds
        left, bottom = max(left, b.left), max(bottom, b.bottom)
        right, top = min(right, b.right), min(top, b.top)
        window = from_bounds(left, bottom, right, top, dataset.transform).round_offsets().round_lengths()
        width = min(out_px, int(window.width))
        height = max(1, round(window.height * width / window.width))
        values = dataset.read(1, window=window, out_shape=(height, width), masked=True,
                              out_dtype="float32", resampling=Resampling.average)
        elevation = np.asarray((values * scale + offset).filled(np.nan), dtype="<f4")
        win_transform = dataset.window_transform(window)
        transform = win_transform * win_transform.scale(window.width / width, window.height / height)
        return elevation, transform, dataset.crs


def show(elevation, title):
    plt.figure(figsize=(5, 5))
    plt.imshow(elevation, cmap="terrain")
    plt.colorbar(label="Elevation (m)")
    plt.title(title)
    plt.show()

## 3. Site catalog and sources
All links come from the official PGDA and PDS pages.

In [ ]:
# Every LOLA 5 m/px site DEM on https://pgda.gsfc.nasa.gov/products/78 (names from that page).
LOLA_5MPP = "https://pgda.gsfc.nasa.gov/data/LOLA_5mpp"
SITE_NAMES = {
    "Site01": "Connecting ridge", "Site04": "Shackleton rim", "Site06": "Nobile rim 1",
    "Site07": "Peak near Shackleton", "Site11": "de Gerlache rim",
    "Site20": "Leibnitz beta plateau", "Site20v2": "Leibnitz beta plateau, extended boundaries",
    "Site23": "Malapert massif", "Site42": "de Gerlache-Kocher massif",
    "Haworth": "Haworth", "Shoemaker": "Shoemaker", "DM1": "Amundsen rim", "DM2": "Nobile rim 2",
    "SL2": "de Gerlache rim", "SL3": "Connecting ridge extension",
    "NPA": "Cabeus exterior wall 1", "NPB": "Amundsen 1", "NPC": "Idel'son L crater 1",
    "NPD": "Malapert crater 1", "LM1": "Shackleton Rim B", "LM2": "Shoemaker Rim A",
    "LM3": "Shoemaker Rim B", "LM4": "Shoemaker Rim C", "LM5": "Shoemaker Rim D",
    "LM6": "Shoemaker Rim E", "LM7": "Faustini Rim A", "LM8": "Shoemaker Rim F",
}


def site_url(site_id):
    return f"{LOLA_5MPP}/{site_id}/{site_id}_final_adj_5mpp_surf.tif"


# Wider, coarser terrain for distant ridges on the horizon: PDS LOLA polar DEM, 80 m/px, 80°S to the pole.
CONTEXT_URL = "https://imbrium.mit.edu/DATA/LOLA_GDR/POLAR/JP2/LDEM_80S_80M.JP2"
CONTEXT_LABEL = "https://imbrium.mit.edu/DATA/LOLA_GDR/POLAR/JP2/LDEM_80S_80M_JP2.LBL"
CONTEXT_SCALE = 0.5  # stored number x 0.5 = height in meters above the 1,737.4 km sphere
CITATION = ("Barker et al. (2021), Planetary and Space Science 203, 105119, "
            "doi:10.1016/j.pss.2020.105119")

unknown = [s for s in SITE_IDS if s not in SITE_NAMES]
assert not unknown, f"Not in the catalog: {unknown}"

if INCLUDE_CONTEXT:
    print(f"LDEM_80S_80M.JP2  {remote_size(CONTEXT_URL) / 1e6:,.1f} MB")
    context_path = download(CONTEXT_URL)
    # The _AUX.XML file holds the scale. GDAL looks for "<file>.aux.xml".
    download(CONTEXT_URL.replace(".JP2", "_AUX.XML"), name=context_path.name + ".aux.xml")

## 4. Process every site
For each site: download it once, average it down to `MAX_PX`, save it, then cut the horizon box around the same center. `average` blends the pixels inside each output pixel, so the terrain stays smooth.

In [ ]:
index = []
for site_id in SITE_IDS:
    url = site_url(site_id)
    print(f"{site_id} ({SITE_NAMES[site_id]}): {remote_size(url) / 1e6:,.1f} MB")
    path = download(url)
    with rasterio.open(path) as dataset:
        site_bounds = dataset.bounds
        source_size = [dataset.width, dataset.height]
    center_x = (site_bounds.left + site_bounds.right) / 2
    center_y = (site_bounds.bottom + site_bounds.top) / 2
    if CROP_KM:
        half = CROP_KM * 500
        box = (center_x - half, center_y - half, center_x + half, center_y + half)
    else:
        box = tuple(site_bounds)

    folder = EXPORT / site_id.lower()
    elevation, transform, crs = read_grid(path, box, MAX_PX)
    meta = save_grid(elevation, polar_metadata(transform, elevation.shape[1], elevation.shape[0], crs, {
        "source": url, "productPage": "https://pgda.gsfc.nasa.gov/products/78",
        "siteId": site_id, "siteName": SITE_NAMES[site_id],
        "sourceResolutionM": 5.0, "sourceSizePx": source_size,
        "sourceScale": 1.0, "sourceOffset": 0.0, "resampling": "average", "citation": CITATION,
    }), folder, f"{site_id.lower()}-height")
    entry = {"siteId": site_id, "siteName": SITE_NAMES[site_id],
             "height": f"{site_id.lower()}/{site_id.lower()}-height.f32",
             "centerLatLon": meta["centerLatLon"], "extentM": meta["extentM"],
             "resolutionM": meta["resolutionM"],
             "minElevationM": meta["minElevationM"], "maxElevationM": meta["maxElevationM"]}
    print(f"  height: {meta['width']}x{meta['height']} at {meta['resolutionM']:.1f} m/px, "
          f"center {meta['centerLatLon']}, {meta['minElevationM']:,.0f} to {meta['maxElevationM']:,.0f} m")

    if INCLUDE_CONTEXT:
        half = CONTEXT_KM * 500
        context, c_transform, c_crs = read_grid(
            context_path, (center_x - half, center_y - half, center_x + half, center_y + half),
            CONTEXT_PX, scale=CONTEXT_SCALE)
        c_meta = save_grid(context, polar_metadata(c_transform, context.shape[1], context.shape[0], c_crs, {
            "source": CONTEXT_URL, "productLabel": CONTEXT_LABEL,
            "siteId": site_id, "role": "horizon context around the site",
            "sourceResolutionM": 80.0, "sourceScale": CONTEXT_SCALE, "sourceOffset": 0.0,
            "resampling": "average",
        }), folder, f"{site_id.lower()}-context")
        entry["context"] = f"{site_id.lower()}/{site_id.lower()}-context.f32"
        entry["contextExtentM"] = c_meta["extentM"]
        print(f"  context: {c_meta['width']}x{c_meta['height']} at {c_meta['resolutionM']:.0f} m/px")

    index.append(entry)
    show(elevation, f"{site_id}: {SITE_NAMES[site_id]}")
    if not KEEP_RAW:
        path.unlink()

(EXPORT / "index.json").write_text(json.dumps({
    "product": "LOLA 5 m/px south pole site DEMs (PGDA product 78)",
    "productPage": "https://pgda.gsfc.nasa.gov/products/78",
    "encoding": "float32-le", "units": "m", "nodataEncoding": "NaN",
    "referenceRadiusM": MOON_RADIUS_M, "sites": index,
}, indent=2) + "\n")
print(f"Wrote {len(index)} sites")

## 5. Zip and download
Unzip into `public/data/` to get `public/data/lola-sites/index.json` and one folder per site.

In [ ]:
zip_path = zip_folder(EXPORT, OUT_ROOT / f"{ASSET_NAME}.zip")
print(f"Exports: {folder_size_mb(EXPORT):.1f} MB in {EXPORT}")
print(f"Zip: {zip_path.stat().st_size / 1e6:.1f} MB at {zip_path}")
if DOWNLOAD_ZIP and IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Download the zip from the file panel (Kaggle: Output tab).")